In [2]:
import pandas as pd
import urllib.parse

# Import Dataset

## Edinburgh

In [3]:
dfe1 = pd.read_csv('dataset/costProfCat-EdinPOI-all.csv', sep=';')
dfe1.head()

,from,to,cost,profit,category
0,1,2,1818.249473,699,Historical
1,1,3,700.407777,3140,Museum
2,1,4,1645.504806,886,Structure
3,1,5,13047.765381,12,Structure
4,1,6,13936.054340,11,Structure


In [4]:
dfe2 = pd.read_csv('dataset/POI-Edin.csv', sep=';')
dfe2['poiName'] = dfe2['poiName'].apply(lambda x: urllib.parse.unquote(x))
dfe2.head()

,poiID,poiName,lat,long,theme
0,1,Edinburgh_Castle,55.948611,-3.200833,Historical
1,2,Holyrood_Palace,55.952500,-3.172500,Historical
2,3,National_Museum_of_Scotland,55.946940,-3.190000,Museum
3,4,Scottish_Parliament_Building,55.952158,-3.175204,Structure
4,5,Forth_Rail_Bridge,56.000421,-3.388726,Structure


In [5]:
dfe3 = pd.read_csv('dataset/userVisits-Edin.csv', sep=';')
dfe3['dateTaken'] = pd.to_datetime(dfe3['dateTaken'], unit='s')
dfe3 = pd.merge(dfe3, dfe2[['poiID', 'poiName']], on='poiID', how='left')
dfe3['Day'] = dfe3['dateTaken'].dt.day_name()
dfe3.head()

,photoID,userID,dateTaken,poiID,poiTheme,poiFreq,seqID,poiName,Day
0,355803498,10063645@N00,2007-01-11 05:15:31,17,Structure,1303,1,"Parliament_House,_Edinburgh",Thursday
1,355803564,10063645@N00,2007-01-11 05:15:49,17,Structure,1303,1,"Parliament_House,_Edinburgh",Thursday
2,355803639,10063645@N00,2007-01-11 05:15:58,17,Structure,1303,1,"Parliament_House,_Edinburgh",Thursday
3,1431486935,10091295@N02,2007-09-22 00:42:26,12,Cultural,1208,2,Usher_Hall,Saturday
4,1432364530,10091295@N02,2007-09-22 00:43:12,12,Cultural,1208,2,Usher_Hall,Saturday


## Glasgow

In [13]:
dfg1 = pd.read_csv('dataset/costProfCat-GlasPOI-all.csv', sep=';')
dfg1.head()

,from,to,cost,profit,category
0,1,2,631.721015,916,Transport
1,1,4,44174.137938,1,Transport
2,1,5,4721.455580,2,Transport
3,1,6,808.589782,48,Transport
4,1,7,1099.186785,125,Education


In [14]:
dfg2 = pd.read_csv('dataset/POI-Glas.csv', sep=';')
dfg2['poiName'] = dfg2['poiName'].apply(lambda x: urllib.parse.unquote(x))
dfg2.head()

,poiID,poiName,lat,long,theme
0,1,Glasgow_Central_railway_station,55.85800,-4.25800,Transport
1,2,Glasgow_Queen_Street_railway_station,55.86220,-4.25120,Transport
2,3,Glasgow_Airport,55.87194,-4.43306,Transport
3,4,Glasgow_Prestwick_International_Airport,55.50944,-4.59444,Transport
4,5,Clyde_Tunnel,55.86867,-4.33115,Transport


In [15]:
dfg3 = pd.read_csv('dataset/userVisits-Glas.csv', sep=';')
dfg3['dateTaken'] = pd.to_datetime(dfg3['dateTaken'], unit='s')
dfg3 = pd.merge(dfg3, dfg2[['poiID', 'poiName']], on='poiID', how='left')
dfg3['Day'] = dfg3['dateTaken'].dt.day_name()
dfg3.head()

,photoID,userID,dateTaken,poiID,poiTheme,poiFreq,seqID,poiName,Day
0,268535960,10063645@N00,2006-10-13 05:07:06,11,Education,1008,1,Glasgow_School_of_Art,Friday
1,284505324,10063645@N00,2006-10-30 22:23:28,2,Transport,916,2,Glasgow_Queen_Street_railway_station,Monday
2,285538933,10063645@N00,2006-10-31 22:26:28,11,Education,1008,3,Glasgow_School_of_Art,Tuesday
3,289494173,10063645@N00,2006-11-05 02:26:35,16,Shopping,848,4,Buchanan_Street,Sunday
4,289494204,10063645@N00,2006-11-05 02:30:17,21,Museum,618,4,Gallery_of_Modern_Art,Sunday


# Visualisasi

## 1. Tempat yang paling sering dikunjungi

### Edinburgh

In [35]:
import plotly.express as px

visit_counts = dfe3.groupby("poiName").size().reset_index(name="visit_count")

visit_counts = visit_counts.sort_values(by="visit_count", ascending=False)

fig = px.bar(
    visit_counts,
    x="poiName",
    y="visit_count",
    title="Tempat yang Paling Sering Diambil Foto",
    labels={"poiName": "Nama Tempat", "visit_count": "Jumlah Foto Diambil"},
    text="visit_count"
)


fig.update_traces(marker_color="orange", textposition="outside")
fig.update_layout(
    xaxis_tickangle=45,
    title=dict(x=0.5),
    yaxis=dict(range=[0, 5600])
)

# Tampilkan plot
fig.show()


### Glasgow

In [39]:
import plotly.express as px

visit_counts = dfg3.groupby("poiName").size().reset_index(name="visit_count")

visit_counts = visit_counts.sort_values(by="visit_count", ascending=False)

fig = px.bar(
    visit_counts,
    x="poiName",
    y="visit_count",
    title="Tempat yang Paling Sering Diambil Foto",
    labels={"poiName": "Nama Tempat", "visit_count": "Jumlah Foto Diambil"},
    text="visit_count"
)


fig.update_traces(marker_color="orange", textposition="outside")
fig.update_layout(
    xaxis_tickangle=45,
    title=dict(x=0.5),
    yaxis=dict(range=[0, 1800])
)

# Tampilkan plot
fig.show()


## 2. Jam berapa suatu tempat paling sering dikunjungi

### Edinburgh

In [41]:
import plotly.express as px


dfe3["dateTaken"] = pd.to_datetime(dfe3["dateTaken"])
dfe3["Hour"] = dfe3["dateTaken"].dt.hour

df_Hall = dfe3[dfe3['poiName'] == "Usher_Hall"]
df_Theatre = dfe3[dfe3['poiName'] == "Edinburgh_Festival_Theatre"]


total_photos_per_hour1 = df_Hall.groupby('Hour').size().reset_index(name="total_photos")
total_photos_per_hour1['location'] = "Usher Hall"
total_photos_per_hour2 = df_Theatre.groupby('Hour').size().reset_index(name="total_photos")
total_photos_per_hour2['location'] = "Edinburgh Festival Theatre"

total_photos_combined = pd.concat([total_photos_per_hour1, total_photos_per_hour2])


fig = px.line(
    total_photos_combined,
    x='Hour',
    y='total_photos',
    color='location',
    title="Total Foto per Jam di Usher Hall dan Edinburgh Festival Theatre",
    labels={"total_photos": "Total Foto", "hour": "Jam", "location": "Lokasi"},
    markers=True
)


fig.update_layout(
    yaxis=dict(range=[0, 250])
)


fig.show()


### Glasgow

In [49]:
import plotly.express as px

dfg3["dateTaken"] = pd.to_datetime(dfg3["dateTaken"])
dfg3["hour"] = dfg3["dateTaken"].dt.hour

df_Street = dfg3[dfg3['poiName'] == "Buchanan_Street"]
df_Lane = dfg3[dfg3['poiName'] == "Ashton_Lane"]

total_photos_per_hour1 = df_Street.groupby('hour').size().reset_index(name="total_photos")
total_photos_per_hour1['location'] = "Buchanan Street"
total_photos_per_hour2 = df_Lane.groupby('hour').size().reset_index(name="total_photos")
total_photos_per_hour2['location'] = "Ashton Lane"


total_photos_combined = pd.concat([total_photos_per_hour1, total_photos_per_hour2])


fig = px.line(
    total_photos_combined,
    x='hour',
    y='total_photos',
    color='location',
    title="Total Foto per Jam di Buchanan Street dan Ashton Lane",
    labels={"total_photos": "Total Foto", "hour": "Jam", "location": "Lokasi"},
    markers=True
)

fig.update_layout(
    yaxis=dict(range=[0, 230])
)

fig.show()

## 3. Pada hari apa suatu tempat sering dikunjungi

### Edinburgh

In [7]:
df_garden1 = dfe3[dfe3['poiName'] == "Princes_Street_Gardens"]
df_garden2 = dfe3[dfe3['poiName'] == "Royal_Botanic_Garden_Edinburgh"]

total_photos1 = df_garden1.groupby("Day").size().reset_index(name="total_photos")
total_photos2 = df_garden2.groupby("Day").size().reset_index(name="total_photos")

In [9]:
import plotly.express as px

df_garden1 = dfe3[dfe3['poiName'] == "Princes_Street_Gardens"]
df_garden2 = dfe3[dfe3['poiName'] == "Royal_Botanic_Garden_Edinburgh"]

total_photos1 = df_garden1.groupby("Day").size().reset_index(name="total_photos")
total_photos2 = df_garden2.groupby("Day").size().reset_index(name="total_photos")

day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
total_photos1["Day"] = pd.Categorical(total_photos1["Day"], categories=day_order, ordered=True)
total_photos1 = total_photos1.sort_values("Day")
total_photos2["Day"] = pd.Categorical(total_photos2["Day"], categories=day_order, ordered=True)
total_photos2 = total_photos2.sort_values("Day")

total_photos1["location"] = "Princes Street Gardens"
total_photos2["location"] = "Royal Botanic Garden Edinburgh"

total_photos_combined = pd.concat([total_photos1, total_photos2])

fig = px.line(
    total_photos_combined,
    x="Day",
    y="total_photos",
    color="location",
    title="Total Foto per Hari di Princes Street Gardens dan Royal Botanic Garden Edinburgh",
    labels={"total_photos": "Total Foto", "Day": "Hari", "location": "Lokasi"},
    markers=True
)

fig.update_layout(
    yaxis=dict(range=[0, 510])
)

fig.show()


### Glasgow

In [24]:
import plotly.express as px

df_gallery1 = dfg3[dfg3['poiName'] == "Kelvingrove_Art_Gallery_and_Museum"]
df_gallery2 = dfg3[dfg3['poiName'] == "Gallery_of_Modern_Art"]

total_photos1 = df_gallery1.groupby("Day").size().reset_index(name="total_photos")
total_photos2 = df_gallery2.groupby("Day").size().reset_index(name="total_photos")

day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
total_photos1["Day"] = pd.Categorical(total_photos1["Day"], categories=day_order, ordered=True)
total_photos1 = total_photos1.sort_values("Day")
total_photos2["Day"] = pd.Categorical(total_photos2["Day"], categories=day_order, ordered=True)
total_photos2 = total_photos2.sort_values("Day")

total_photos1["location"] = "Kelvingrove Art Gallery and Museum"
total_photos2["location"] = "Gallery of Modern Art"
total_photos_combined = pd.concat([total_photos1, total_photos2])


fig = px.line(
    total_photos_combined,
    x="Day",
    y="total_photos",
    color="location",
    title="Total Foto per Hari di Kelvingrove Art Gallery and Museum dan Gallery of Modern Art",
    labels={"total_photos": "Total Foto", "Day": "Hari", "location": "Lokasi"},
    markers=True
)


fig.update_layout(
    yaxis=dict(range=[0, 260])
)

fig.show()